In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# The shape of the old population

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/shape.ipynb)

Every number on this page is written by `scripts/paper_text_shape.py` from the result tables (`results/macros_shape.tex`); the code
cells are the figure scripts themselves (`scripts/paper_shape_*.py`), step by step.

The kinematic model of the Letter puts every star in one thin plane. The old population is not thin: the vertical velocity
dispersion $\sigma_z$ of the most metal-poor giants is 32 km/s, comparable to their rotation. Here 'old' and 'metal-poor' refer to the
iron abundance [Fe/H] (the logarithm of the iron-to-hydrogen ratio relative to the Sun's); the giants are divided into five bins of
[Fe/H], as in the Letter. Its three-dimensional shape tells us whether a thin-disc model can describe it. The same
old, metal-poor population is seen through two tracers. The giants have three-dimensional velocities; through the tensor virial
theorem they give the flattening that a steady population with these motions must have. The RR Lyrae have distances; their depth
along the line of sight gives the flattening directly, if their geometry can be modelled.

## Flattening implied by the motions

For a steady, axisymmetric population of tracer stars in a fixed potential $\Phi$, with $z$ along the symmetry axis, the tensor
virial theorem gives $2K_{ij}+W_{ij}=0$ with $K_{ij}=\frac{1}{2}\sum v_iv_j$ and $W_{ij}=-\sum x_i\,\partial\Phi/\partial x_j$,
where $x_i$ and $v_i$ ($i,j=x,y,z$) are the position and velocity
components of each tracer star and the sums run over the tracer's stars ([Binney & Tremaine 2008](https://ui.adsabs.harvard.edu/abs/2008gady.book.....B), section 4.8.3). The ratio
of its in-plane to its vertical components,

$$
\Omega_{\rm kin}\equiv\frac{\sum(v_x^2+v_y^2)}{2\sum v_z^2}=\frac{W_{xx}+W_{yy}}{2W_{zz}}\equiv\Omega_{\rm pot}(q),
$$

ties the kinematics on the left to the shape of the tracer on the right. Stars that rotate fast and move little up and down must lie
in a flattened distribution; the more vertical motion they have for their rotation, the rounder it must be. The axis ratio $q=c/a$, of the short
(vertical) axis $c$ to the long axis $a$ of the tracer's density, measures this: $q=1$ is a sphere and $q=0$ a thin disc. The old stars are a small part of the LMC's mass, so this is the theorem for
a tracer, not the self-gravitating relation between $v/\sigma$ and ellipticity. We realise each giant bin's kinematic model,
fitted to its velocities in the kinematics chapter (rotation curve, velocity ellipsoid and its radial gradient), on an oblate
spheroid with density $\propto K_0(m/h)$. Here $R$ and $z$ are the cylindrical radius and height in the disc frame,
$m^2=R^2+z^2/q^2$ is the spheroidal radius, and $K_0$ is the modified Bessel function of the second kind, chosen so that the face-on
profile is exponential, with the bin's scale length $h$ (fitted to its surface density of giants, weighted for the selection of the
SDSS-V BOSS spectra), and find the axis ratio $q$ at which the equation holds. The whole tracer is integrated, not only
the radii the fits constrain. The potential is not measured here, so we use three that bracket the LMC's circular speed $v_c$
(65–90 km/s at 2–8 kpc). P1 is a cored logarithmic halo,
$\Phi=\tfrac12V^2\ln(R_{\rm c}^2+R^2+z^2/q_\Phi^2)$ with asymptotic circular speed $V$ = 92 km/s, core radius $R_{\rm c}$ = 2 kpc and
$q_\Phi=1$; P2 is the same halo flattened to $q_\Phi$ = 0.85; P3 is a [Miyamoto & Nagai (1975)](https://doi.org/10.1093/pasj/27.4.533)
disc of mass 2.7 × 10⁹ M☉, scale length $a_{\rm MN}$ = 1.6 kpc and scale height $b_{\rm MN}$ = 0.3 kpc in a spherical
cored logarithmic halo ($V$ = 80 km/s, $R_{\rm c}$ = 3 kpc). The uncertainty within each
potential comes from 200 joint posterior draws of the kinematic fit, each with $h$ drawn within its bootstrap range.

### Solving for the axis ratio

`scripts/shape_virial.py` does this for every giant bin and for the RR Lyrae. It takes the posterior samples of each group's
kinematic fit and its scale length $h$, realises the posterior-median model and each joint draw as a tracer, and finds $q$ in each
of P1, P2 and P3. Every case is run twice: once with the dispersions following the fitted radial gradient all the way in, and once
with them held constant inside the radius below which the fits do not constrain them. For the posterior-median model it also
records $2K/|W|$, the test of the depth of the potential described below. Its table, `results/shape_virial.csv`, feeds
Table 8.1, Figure 8.1 and the numbers of this section.

<details><summary>About <code>scripts/shape_virial.py</code></summary>

```text
Flattening of each kinematic group implied by its rotation and dispersions (paper Section 5; lmcold.virial; KB chapter 'The shape
of the old population', section 'Flattening implied by the motions' and Table shape_q).

For every giant [Fe/H] bin (stage-1 ellipsoid fits at 2-9 deg, data/kin_mcmc_data_ellipsoid_samples.npz) and the RR Lyrae (stage-1
fit at 4-9 deg, data/kin_mcmc_data_ellipsoid_outer_samples.npz, whose dispersions the paper quotes): the kinematic model and the group's
exponential scale length h (results/kin_vc_drift_h.csv) are realised as a round tracer, and the axis ratio q at which the potential
side of the tensor virial theorem equals the kinetic side is found in each potential of lmcold.virial.POTENTIALS.
Uncertainty: NDRAW joint posterior draws (all parameters of one draw together), each with h drawn from a Gaussian of its bootstrap
16-84 half-width. Central extrapolation: the fits constrain the dispersions at R >~ 2 kpc; the exponential gradient carries them
inward, where much of the tracer's kinetic energy lies, so every case is also run with the dispersions held constant inside
R_FLOOR (lmcold.spheroid.velocities).

Absolute balance: the ratios do not test the depth of the potential, so 2K/|W| (lmcold.virial.balance) of the posterior-median model
realised at its q is also given (1 if the potential supports the measured motions).

Reads: the posterior samples data/kin_mcmc_data_ellipsoid_samples.npz and kin_mcmc_data_ellipsoid_outer_samples.npz (Zenodo bundle 1;
scripts/fit_kinematics_mcmc.py) and results/kin_vc_drift_h.csv (scale lengths h with bootstrap 16th and 84th percentiles [kpc],
scripts/kin_vc_drift.py).
Writes results/shape_virial.csv (population, potential, r_floor, h, omega_kin, q and balance of the posterior-median vector, q_p16,
q_p50, q_p84 over the draws with a solution, f_none: draws with no solution, f_prolate: draws with q > 1), read by
scripts/paper_shape_virial.py, paper_table_shape.py, paper_text_shape.py and shape_rrl_cells.py.

Run: .venv/bin/python scripts/shape_virial.py
```
</details>

*Shown, not run here: the posterior samples of the kinematic fits (Zenodo bundle 1); one solution for q per posterior draw, group, potential and central floor. The cells below are the script itself; run it from the repository with* `python scripts/shape_virial.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The number of posterior draws per group and the radius inside which the dispersions are held constant in the second set of runs.

```python
import re

import numpy as np
import pandas as pd

from lmcold import kinfit, spheroid, virial
from lmcold.sources import ROOT

NDRAW = 200  # joint posterior draws per group
R_FLOOR = 2.0  # kpc: dispersions held constant inside this radius (the fits constrain them at R >~ 2 kpc)
P = kinfit.PARAMS  # parameter names of a kinematic model vector
```

#### Posterior draws
`draws` reads the stage-1 posterior samples of one population and returns its posterior-median model and `NDRAW` joint draws, each a
full parameter vector of `lmcold.kinfit`.

```python
def draws(npz, pop, rng):
    """Posterior-median vector and NDRAW joint draws (kinfit vectors) of population index pop, stage 1, from a samples file.

    Parameters
    ----------
    npz : str
        File name in data/ of the posterior samples (keys 'stage|population|parameter[_k]', and 'names', the population labels).
    pop : int
        Population index in that file (0-4: the giant [Fe/H] bins; 5: the RR Lyrae in the outer-fit file).
    rng : numpy.random.Generator
        Picks the draws (sample indices, the same index for all parameters of one draw).

    Returns
    -------
    (str, numpy.ndarray, list of numpy.ndarray)
        Population label, posterior-median vector (len(kinfit.PARAMS); parameters absent from the file keep kinfit.start()), and the
        NDRAW draw vectors. Units as kinfit (deg, km/s, kpc).
    """
    z = np.load(ROOT / 'data' / npz, allow_pickle=True)
    # parameter name -> key, for the stage-1 keys '1|pop|name' of this population (a '_k' suffix of vector parameters removed)
    keys = {re.sub(r'_\d+$', '', k.split('|')[2]): k for k in z.files if k.startswith(f'1|{pop}|')}
    # keep only the parameters of kinfit.PARAMS
    keys = {n: k for n, k in keys.items() if n in P}
    flat = {n: z[k].reshape(-1) for n, k in keys.items()}  # parameter name: all samples (chains flattened)
    # start from kinfit.start() so parameters absent from the file keep their default values
    med, out = kinfit.start(), []
    # posterior median of each sampled parameter
    for n, v in flat.items():
        med[P.index(n)] = np.median(v)
    # NDRAW random sample indices; each draw takes all parameters from the same sample
    for i in rng.integers(0, len(next(iter(flat.values()))), NDRAW):
        p = kinfit.start()
        for n, v in flat.items():
            p[P.index(n)] = v[i]
        out.append(p)
    return str(z['names'][pop]), med, out
```

#### Load the data
The scale lengths $h$ of the groups, and the posterior draws of the five giant bins (2-9 deg fit) and of the RR Lyrae (4-9 deg fit).

```python
# fixed seed: draws and realisations are reproducible
rng = np.random.default_rng(5)
H = pd.read_csv(ROOT / 'results/kin_vc_drift_h.csv', index_col=0)  # h, h_p16, h_p84 [kpc] per population
# the five giant [Fe/H] bins (indices 0-4) of the 2-9 deg fit; the RR Lyrae (index 5) of the 4-9 deg fit
groups = [draws('kin_mcmc_data_ellipsoid_samples.npz', i, rng) for i in range(5)]
groups.append(draws('kin_mcmc_data_ellipsoid_outer_samples.npz', 5, rng))
```

#### The axis ratio in each potential
For each group, with and without the central floor on the dispersions, and in each potential P1-P3: $q$ of the posterior-median
model (`lmcold.virial.flattening`), $2K/|W|$ of that model realised at its $q$, and $q$ for each posterior draw with $h$ drawn
from a Gaussian of half the bootstrap 16th-84th range (floored at 0.3 kpc). Draws whose kinematics no tracer on the $q$ grid
matches are counted in `f_none`; those with $q>1$ (elongated along the symmetry axis) in `f_prolate`.

```python
rows = []
for pop, med, ds in groups:
    # scale length and its bootstrap 16th/84th percentiles [kpc]
    h, hlo, hhi = H.loc[pop, ['h', 'h_p16', 'h_p84']]
    # without (0) and with the constant-dispersion floor inside R_FLOOR [kpc]
    for r_floor in (0.0, R_FLOOR):
        for name, grad in virial.POTENTIALS.items():
            q, om = virial.flattening(med, h, grad, rng, r_floor)  # q and Omega_kin of the posterior-median model
            X = spheroid.spheroid(h, q, virial.NREAL, rng) if np.isfinite(q) else None  # positions (NREAL, 3) [kpc] at that q
            bal = virial.balance(X, spheroid.velocities(med, X, rng, r_floor), grad) if X is not None else np.nan  # 2K/|W|
            # q for each draw, with h drawn from a Gaussian of sigma = half the 16th-84th range, floored at 0.3 kpc
            qd = np.array([virial.flattening(d, max(h + 0.5 * (hhi - hlo) * rng.normal(), 0.3), grad, rng, r_floor)[0] for d in ds])
            ok = np.isfinite(qd)  # draws with a solution
            # 16th, 50th, 84th percentiles of q over the draws with a solution
            pc = np.percentile(qd[ok], [16, 50, 84]) if ok.any() else [np.nan] * 3
            rows.append(dict(population=pop, potential=name, r_floor=r_floor, h=h, omega_kin=om, q=q, balance=bal, q_p16=pc[0], q_p50=pc[1],
                             # f_prolate: fraction of all draws with q > 1
                             q_p84=pc[2], f_none=1 - ok.mean(), f_prolate=np.mean(qd[ok] > 1) * ok.mean()))
            print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in rows[-1].items()}, flush=True)
pd.DataFrame(rows).to_csv(ROOT / 'results/shape_virial.csv', index=False, float_format='%.4f')
```

### The axis ratio of each giant bin

Table 8.1 and Figure 8.1 give the result. The most metal-poor giants need $q$ = 0.57, 0.49 and 0.50 in P1, P2 and
P3 (posterior 16th–84th percentiles ±0.02); $q$ falls steadily with metallicity, to 0.25–0.38 above [Fe/H] = -0.5. Even the
oldest, most metal-poor stars therefore form a thick, rotating disc about half as thick as it is wide, not a round halo, and the
younger, more metal-rich groups are flatter still. The choice of potential moves $q$ by 0.08 in the most metal-poor bin, about
4 times its posterior half-width, and by up to 0.13 in the most metal-rich, and is the main uncertainty. It also fixes the
depth of the potential, which the ratio in the equation does not test: the trace of the theorem, $2K/|W|$ with $K=\sum_iK_{ii}$ and
$W=\sum_iW_{ii}$, which equals 1 for a tracer in equilibrium, is 0.96–1.07 in P1
and P2 for the bins below [Fe/H] = -0.5 (1.15 above), but 1.08–1.19 in P3, which is too shallow for the measured
motions.

In [ ]:
import pandas as pd
# Virial flattening q per giant bin and potential, without the constant-dispersion floor inside R_FLOOR (r_floor = 0; scripts/shape_virial.py)
v = pd.read_csv('results/shape_virial.csv')
v = v[(v.r_floor == 0) & v.population.str.startswith('giants')]
v['q (16th–84th)'] = v.apply(lambda r: f'{r.q:.2f} ({r.q_p16:.2f}–{r.q_p84:.2f})', axis=1)
# One row per population, one column per potential; last column: virial ratio 2K/|W| in each potential
t = v.pivot(index='population', columns='potential', values='q (16th–84th)').reindex(v.population.unique())
t['2K/|W| (P1, P2, P3)'] = v.groupby('population', sort=False).balance.apply(lambda b: ', '.join(f'{x:.2f}' for x in b))
t.index.name = None; t.columns.name = None
t

**Table 8.1.** Axis ratio $q=c/a$ implied by the kinematics of each giant bin: posterior-median model, with the 16th–84th
percentiles of 200 joint draws in parentheses, in the three potentials P1, P2 and P3, and $2K/|W|$ of the median model at that
$q$ (`results/shape_virial.csv`, written by `scripts/shape_virial.py`).

<details><summary>About <code>scripts/paper_shape_virial.py</code></summary>

```text
Paper figure (Section 5): the flattening each giant [Fe/H] bin's kinematics imply, q = c/a of an oblate K0 tracer from the tensor
virial theorem (scripts/shape_virial.py, results/shape_virial.csv, no central floor), against the bin's median [Fe/H]
(lmcold.samples.giants), in the three potentials of lmcold.virial (posterior-median vector: markers; 16th-84th percentiles of the joint
posterior draws: bars). Points of P1 and P3 are offset in [Fe/H] by -DX and +DX for legibility. The grey band is the RR Lyrae (same old
population; median [Fe/H] from results/kin_support_ellipsoid.csv): q from their near-to-far distance gradient at the metal-poor giants' inclination (results/
shape_rrl_depth.csv: bootstrap 16-84 per cent, darker; and the range from a +-0.02 mag coherent modulus gradient, lighter).

Writes figures/paper_shape_virial.png (KB chapter 'The shape of the old population', Figure paper_shape_virial).

Run: .venv/bin/python scripts/paper_shape_virial.py   (after scripts/shape_virial.py and scripts/shape_rrl_cells.py)
```
</details>

**Data read by `scripts/paper_shape_virial.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/shape_rrl_depth.csv` | repository | axis ratio from the RR Lyrae depth gradient at the inclination of the motions; written by `python scripts/shape_rrl_cells.py` |
| `results/shape_virial.csv` | repository | axis ratio implied by the motions (tensor virial theorem) per giant bin and potential; written by `python scripts/shape_virial.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Plotting choices: the offset of the P1 and P3 points in [Fe/H], and the marker, colour and legend label of each potential.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold import samples
from lmcold.layout import panel_grid
from lmcold.sources import ROOT

DX = 0.025  # dex, horizontal offset of the points: (k - 1) DX for the k-th potential, so P1 at -DX, P2 at 0, P3 at +DX
STYLE = {'P1': ('o', 'k'), 'P2': ('s', '#2166ac'), 'P3': ('D', '#b2182b')}  # potential: (marker, colour)
# Legend label of each potential of lmcold.virial.
LABELS = {'P1': 'P1: cored logarithmic halo', 'P2': r'P2: P1 flattened, $q_\Phi=0.85$', 'P3': 'P3: disc + halo'}

#### Load the data
The axis ratios implied by the motions from `results/shape_virial.csv` (`scripts/shape_virial.py`), the RR Lyrae flattening
from their depth from `results/shape_rrl_depth.csv` (`scripts/shape_rrl_cells.py`), the median [Fe/H] of the RR Lyrae from
`results/kin_support_ellipsoid.csv`, and the giant catalogues of Zenodo bundle 1 through `lmcold.samples.giants()`, split into the
five [Fe/H] bins (for the median [Fe/H] of each bin).

In [ ]:
v = pd.read_csv(ROOT / 'results/shape_virial.csv')  # q per population, potential and central floor r_floor
d = pd.read_csv(ROOT / 'results/shape_rrl_depth.csv')  # RR Lyrae q from the distance gradient, per case (bootstrap, dipole)
# Median [Fe/H] of the RR Lyrae members.
feh_rrl = pd.read_csv(ROOT / 'results/kin_support_ellipsoid.csv').set_index('population').feh_med['RR Lyrae']
giants = samples.giants()  # (label, catalogue dict) per [Fe/H] bin, from data/giants_apogee.parquet and giants_boss.parquet

#### The axis ratios implied by the motions
`results/shape_virial.csv` holds, for each giant [Fe/H] bin and each of the three potentials, the axis ratio $q$ of the tracer
whose tensor-virial ratio matches the bin's kinematics: posterior-median model and the 16th-84th percentiles of the posterior
draws. We use the runs without a central dispersion floor, and the median [Fe/H] of each bin.

In [ ]:
v = v[(v.r_floor == 0) & v.population.str.startswith('giants')]
# Median [Fe/H] [dex] of each giant bin, keyed by the population name of the csv.
feh = {f'giants {lab}': np.median(c['feh']) for lab, c in giants}

#### The RR Lyrae: flattening from their depth
`results/shape_rrl_depth.csv` gives $q$ from the near-to-far distance gradient of the RR Lyrae at the inclination of the
metal-poor giants: bootstrap resamples, and the two refits with a coherent distance error of $\pm0.02$ mag across the line of
nodes.

In [ ]:
# Bootstrap values of the RR Lyrae q (dimensionless).
b = d[d.case == 'bootstrap'].q

#### The figure
Grey bands: the RR Lyrae (light: the distance-error range; dark: bootstrap 16th-84th percentiles). Markers: the giants in the
three potentials, offset slightly in [Fe/H] for legibility.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
# One panel; legend above it.
fig, ax = panel_grid(1, 1, width=3.5, left=0.55, right=0.1, top=0.6)
a = ax[0, 0]
# RR Lyrae bands, 0.1 dex wide about their median [Fe/H]: light = range of the two dipole refits, dark = bootstrap 16th-84th
a.add_patch(plt.Rectangle((feh_rrl - 0.05, d[d.case.str.startswith('dipole')].q.min()), 0.1,
                          np.ptp(d[d.case.str.startswith('dipole')].q), color='0.85', lw=0))
a.add_patch(plt.Rectangle((feh_rrl - 0.05, b.quantile(0.16)), 0.1, b.quantile(0.84) - b.quantile(0.16), color='0.6', lw=0,
                          label='RR Lyrae: distance gradient'))
for k, (pot, (mk, c)) in enumerate(STYLE.items()):
    # The bins in this potential: median-vector q with 16th-84th percentile bars, at the bin's median [Fe/H] offset by (k - 1)
    # DX.
    g = v[v.potential == pot]
    x = np.array([feh[p] for p in g.population]) + (k - 1) * DX
    a.errorbar(x, g.q, yerr=[g.q - g.q_p16, g.q_p84 - g.q], fmt=mk, color=c, ms=4, capsize=0, lw=1, label=LABELS[pot])
# Axis ranges; the legend above the panel; the plotted values printed.
a.set(xlabel='[Fe/H]', ylabel='axis ratio $q=c/a$ from rotation', ylim=(0, 0.8), xlim=(-2.0, -0.2))
a.legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=6, frameon=False, ncol=2)
fig.savefig(ROOT / 'figures/paper_shape_virial.png')
print(v[['population', 'potential', 'q', 'q_p16', 'q_p84', 'balance']].round(3).to_string(index=False))

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_shape_virial.png')

**Figure 8.1.** Axis ratio $q$ implied by the motions of each giant bin, against its median [Fe/H] (dex), in the
potentials P1 (black), P2 (blue) and P3 (red); bars: 16th–84th percentiles of the posterior. The spread between the potentials, not the bars, is the real uncertainty: the
most metal-poor stars need $q\approx0.5$ and the flattening grows with metallicity. Grey: the RR Lyrae (median [Fe/H] = -1.38),
$q$ from their near-to-far distance gradient at the inclination of the metal-poor giants (dark: bootstrap 16th–84th percentiles;
light: a coherent ±0.02 mag distance error across the line of nodes; see Figure 8.2). `scripts/paper_shape_virial.py`

We tested the procedure on simulated analogues. For the two metal-poor bins of the 8 zoom-in simulated galaxies with all particle types,
from Auriga ([Grand et al. 2017](https://doi.org/10.1093/mnras/stx071)) and FIRE-2 ([Hopkins et al. 2018](https://doi.org/10.1093/mnras/sty1690)), the same steps (the kinematic model fitted to the mock giants, realised as a $K_0$ spheroid) recover the true axis
ratio of the particles (0.49–0.60) to -0.02 (16th–84th percentiles -0.04 to 0.00, worst -0.08) when the
analogue's own potential is used. With P1, P2 or P3 instead the offsets are +0.13, +0.03 and +0.08: the method is
accurate, and its error is the potential. Holding the dispersions constant inside 2 kpc, where the fits do not constrain them,
changes $q$ by at most 0.017. The bisymmetric ($m=2$) streaming of the fits, which the realisation omits, adds at most
1.2 per cent to the in-plane kinetic energy and lowers $q$ by at most 0.004 (posterior-median
models of the giant bins, all three potentials). For the RR Lyrae the theorem gives no answer: their $\sigma_z$ is not measured, and in most posterior draws no oblate
tracer has their kinematics.

## The depth of the RR Lyrae

The RR Lyrae offer two geometric clues to the same shape: how much nearer one side of the LMC is than the other, and how oval the
population looks on the sky. Each clue depends on both the tilt and the flattening, so each alone allows a range of shapes; the tilt
from the motions picks one. The RR Lyrae at angular distances 2° < ρ < 7° from the LMC centre (13 322 stars) give both clues. We count them in 192 sky cells
(8 rings, 24 sectors) and take the median distance modulus of each cell; the scatter of the moduli, and with it the poorly
known per-star noise of the distances (from the period–Wesenheit relation; the Wesenheit magnitude is a combination of $V$ and $I$
that is insensitive to dust), does not enter. For an oblate body whose symmetry axis is inclined by $i$ to the line of sight ($i=0$
face-on), with its line of nodes (where its mid-plane crosses the sky plane) at position angle $\theta$ (east of north), these cells
measure two numbers:
the axis ratio of its outline on the sky, $q_{\rm sky}^2=\cos^2i+q^2\sin^2i$, from the counts, and the gradient of its mean depth
across the line of nodes, $g=\sin i\cos i\,(1-q^2)/q_{\rm sky}^2$, from the medians (a thin disc has $g=\tan i$, a sphere $g=0$).
The RR Lyrae give $q_{\rm sky}$ = 0.84 and $g$ = 0.36. Each number alone allows a curve of $(i,q)$, and together they would fix the
shape at $i$ = 48°, $q$ = 0.69, with the line of nodes at 165°.

That solution is not the tilt of the LMC. The oblate model fits the cells poorly ($\chi^2$ = 1067 for the counts and 522
for the medians, each over 192 cells), and what it leaves is structured (Figure 8.4): the counts are lopsided, with an
amplitude $A_1$ of the first azimuthal harmonic ($m=1$, relative to the mean count of the ring) of 45–58 per cent at 5–7° (an excess to the north-east and a deficit, with Pearson
residuals $(n-\lambda)/\sqrt\lambda$ of the observed counts $n$ about the model counts $\lambda$ down to -5.4, to the west and south-west, the side on which the outer disc is truncated;
[Mackey et al. 2018](https://doi.org/10.3847/2041-8213/aac175)), and the OGLE-IV survey footprint covers at least 98.6 per cent of every ring. Above
all, no oblate body inclined by the 25° of the motions can look as elongated as the RR Lyrae do, since
$q_{\rm sky}\geq\cos i$ = 0.91. The elongation on the sky is an elongation in the plane, as found for the LMC's stars
([van der Marel 2001](https://doi.org/10.1086/323100)) and for its RR Lyrae (a triaxial ellipsoid of 6.5:4.6:3.7 kpc with
$i=22\pm4°$ and $\theta=167\pm7°$; [Cusano et al. 2021](https://doi.org/10.1093/mnras/stab901)); it also turns the density's line
of nodes (165°) away from the kinematic one (144°). A mock at the geometry of the metal-poor giants ($i$ = 24.9°,
$\theta$ = 144°) with an in-plane axis ratio $b/a$ = 0.85 and $c/a$ = 0.41, fitted with the oblate model, gives
$i$ = 50°, $q$ = 0.71, $\theta$ = 152°, $q_{\rm sky}$ = 0.84 and $g$ = 0.34, close to the data (arrow in Figure 8.2).

The depth gradient is the part that survives. Held at the inclination of the motions, the medians give $q$ = 0.41 (bootstrap
16th–84th percentiles 0.39–0.46). A coherent distance error of ±0.02 mag across the line of nodes, the size of the
cell-to-cell scatter the model leaves (0.028 mag), moves this to 0.19–0.54. The RR Lyrae therefore confirm that the old
population is thick and flattened, with $q$ below about 0.54, consistent with the 0.46–0.57 its motions imply, but they do
not measure $q$ better than that. Figure 8.2 shows the two clues in the plane of $i$ and $q$.

### Fitting the sky cells

`scripts/shape_rrl_cells.py` makes the fits of this section. It selects the RR Lyrae with a Wesenheit distance inside the footprint,
counts them in the sky cells with the median distance modulus of each cell, and fits the inclined oblate model with all parameters
free. It then refits with the inclination held at each value of a grid, which gives the best $q$ at every inclination (the 'ridge'),
and reads the ridge at the inclination of the motions. At the inclination of the metal-poor giants it repeats the fit for bootstrap
resamples of the stars and with the coherent distance error added with either sign. Its two tables, `results/shape_rrl_cells.csv` and
`results/shape_rrl_depth.csv`, are read by the three figure scripts of this chapter.

<details><summary>About <code>scripts/shape_rrl_cells.py</code></summary>

```text
Flattening of the old, metal-poor population from its density (RR Lyrae sky counts and mean distances, without the noise floor)
against its inclination and rotation-implied flattening from its motions (paper Section 5; KB chapter 'The shape of the old
population', section 'The depth of the RR Lyrae').

The RR Lyrae (data/rrab.parquet members with an OGLE Wesenheit modulus, in the 2-7 deg footprint of lmcold.shape) are summarised in
sky cells (lmcold.shape.cell_stats: counts and median moduli; the scatter of the moduli, and so the noise floor, does not enter) and
fitted with the inclined oblate K0 spheroid (lmcold.shape.fit_cells). The cells measure two numbers (lmcold.shape.projected): the
outline axis ratio on the sky, q_sky (counts), and the near-to-far depth gradient, g (medians). Each is a curve in (inclination, q);
they cross at the free fit. The ridge (best q at each inclination of INC_GRID) gives q at the inclination of the same population
from its motions: the metal-poor giant bins OLD ([Fe/H] < -1.1; the RR Lyrae median is -1.38), stage-1 ellipsoid fits at 2-9 deg
(results/kin_mcmc_data_ellipsoid.csv), and, less precise, the RR Lyrae proper motions at 4-9 deg (..._outer.csv, as Section 4).
Compared with the rotation-implied q of the OLD bins (results/shape_virial.csv, three potentials).

q at the giants' inclination, its bootstrap scatter (NBOOT resamples of the stars) and its sensitivity to a coherent modulus
gradient across their line of nodes (DIPOLE mag at the footprint edge, both signs), the inclination held in each refit.

Reads: data/rrab.parquet (Zenodo bundle 1); results/kin_mcmc_data_ellipsoid.csv and kin_mcmc_data_ellipsoid_outer.csv (posterior
summaries of the kinematic fits, scripts/fit_kinematics_mcmc.py); results/shape_virial.csv (scripts/shape_virial.py, printed for
comparison only).
Writes results/shape_rrl_cells.csv (the free fit and the ridge: inclination, theta, q, h, delta, lnL, dlnL) and results/
shape_rrl_depth.csv (case: data, dipole +-DIPOLE, bootstrap; q, theta at the giants' inclination); the figure is made by
scripts/paper_shape_rrl.py. The tables are read by scripts/paper_shape_virial.py, paper_shape_rrl.py, paper_shape_rrl_residuals.py
and shape_rrl_triaxial.py; the last three also import OLD, kin_inc, motion_incs or virial_q from this file.

Run: .venv/bin/python scripts/shape_rrl_cells.py
```
</details>

*Shown, not run here: the RR Lyrae catalogue data/rrab.parquet (Zenodo bundle 1); repeated fits of the sky cells along a grid of inclinations and for bootstrap resamples. The cells below are the script itself; run it from the repository with* `python scripts/shape_rrl_cells.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The grid of inclinations of the ridge, the number of bootstrap resamples, the size of the coherent distance error, and the two
metal-poor giant bins whose motions give the old population's tilt.

```python
import numpy as np
import pandas as pd

from lmcold import shape
from lmcold.sources import ROOT

INC_GRID = np.arange(15.0, 60.1, 2.5)  # deg: inclinations 15-60 deg in steps of 2.5 deg at which the ridge fits hold inc fixed
NBOOT = 50  # bootstrap resamples of the stars for q at the kinematic inclination
DIPOLE = 0.02  # mag: coherent modulus offset at rho = 7 deg across the kinematic line of nodes (+ and -)
OLD = ('giants [Fe/H] < -1.5', 'giants -1.5 < [Fe/H] < -1.1')  # the metal-poor giant bins, the RR Lyrae's metallicities
```

#### Helpers
`kin_inc` reads the inclination (or another parameter) of given populations from a table of kinematic posterior summaries;
`motion_incs` gives the old population's tilt from the motions of the metal-poor giants and of the RR Lyrae; `virial_q` reads the
axis ratio the motions of the metal-poor giants require.

```python
def kin_inc(name, populations=('RR Lyrae',), param='inc'):
    """Stage-1 inclination (or another param) [deg] of the given populations in results/{name}.csv: mean of their medians, lowest p16,
    highest p84.

    Parameters
    ----------
    name : str
        Stem of the posterior-summary table in results/ (columns stage, population, param, median, p16, p84).
    populations : tuple of str
        Population labels of that table to combine.
    param : str
        Parameter name, e.g. 'inc' (inclination) or 'theta' (line of nodes, east of north) [deg].

    Returns
    -------
    (float, float, float)
        Mean of the populations' posterior medians, the lowest 16th and the highest 84th percentile.
    """
    # Stage-1 (own-fit) rows of the requested populations and parameter.
    t = pd.read_csv(ROOT / f'results/{name}.csv')
    t = t[(t.stage == 1) & t.population.isin(populations) & (t.param == param)]
    return t['median'].mean(), t.p16.min(), t.p84.max()  # (median, p16, p84) [deg]


def motion_incs():
    """The old population's inclination from its motions: {label: (median, p16, p84)} [deg], the metal-poor giants first (the two OLD
    bins of the 2-9 deg ellipsoid fit), then the RR Lyrae proper motions (the 4-9 deg fit, results/kin_mcmc_data_ellipsoid_outer.csv)."""
    # Giants: the two OLD bins of the 2-9 deg ellipsoid fit combined; RR Lyrae: their own 4-9 deg fit.
    return {'metal-poor giants': kin_inc('kin_mcmc_data_ellipsoid', OLD),
            'RR Lyrae proper motions': kin_inc('kin_mcmc_data_ellipsoid_outer')}


def virial_q():
    """Rotation-implied q of the OLD bins (results/shape_virial.csv, no central floor), all potentials: a DataFrame with one row per
    bin and potential (columns population, potential, q, q_p16, q_p84, ...)."""
    v = pd.read_csv(ROOT / 'results/shape_virial.csv')
    # OLD bins only, the variant without a central floor (r_floor = 0).
    return v[v.population.isin(OLD) & (v.r_floor == 0)]
```

#### Load the data
The RR Lyrae members with a distance modulus from the OGLE Wesenheit magnitude (`dmu_src == 'I'`) inside the footprint of
`lmcold.shape` (2-7 deg from the LMC centre, within the modulus window), counted in the sky cells: number of stars, median modulus
and its error per cell. The kinematic tables are read later through `motion_incs()` and `kin_inc()`.

```python
t = pd.read_parquet(ROOT / 'data/rrab.parquet')  # RR Lyrae: ra, dec [deg], membership, relative distance modulus dmu [mag], its source dmu_src
t = t[t.member & (t.dmu_src == 'I')]  # members with an OGLE I-band Wesenheit modulus
# Footprint of lmcold.shape: 2-7 deg from the centre, modulus within its window.
t = t[shape.in_footprint(t.ra.to_numpy(), t.dec.to_numpy(), t.dmu.to_numpy())]
stats = shape.cell_stats(t.ra.to_numpy(), t.dec.to_numpy(), t.dmu.to_numpy())  # per cell: counts, median dmu, its error
```

#### The free fit and the ridge
The oblate model with all five parameters free (inclination, line of nodes, axis ratio $q$, scale length $h$, modulus zero point $\delta$), then
refitted with the inclination held at each value of `INC_GRID`: the best $q$ at each inclination. `dlnL` is the log likelihood
relative to the best of all these fits.

```python
rows = []  # one dict per fit: fit type, lnL and the CELL_PARAMS (inc [deg], theta [deg], q, h [kpc], delta [mag])
# Free fit: all five parameters; best is the parameter vector, llb its log likelihood.
best, llb = shape.fit_cells(stats)
rows.append(dict(fit='free', lnL=llb, **dict(zip(shape.CELL_PARAMS, best))))
# Ridge: refit at each inclination of INC_GRID, each started from the free fit.
start = best
for inc in INC_GRID:
    x, ll = shape.fit_cells(stats, fixed={'inc': inc}, start=start)
    rows.append(dict(fit='ridge', lnL=ll, **dict(zip(shape.CELL_PARAMS, x))))
    print(rows[-1], flush=True)
R = pd.DataFrame(rows)
R['dlnL'] = R.lnL - R.lnL.max()  # log likelihood relative to the best fit
R.to_csv(ROOT / 'results/shape_rrl_cells.csv', index=False, float_format='%.4f')
```

#### q at the tilt from the motions
The ridge interpolated at the inclination of the metal-poor giants and at that of the RR Lyrae proper motions (median and
16th-84th percentiles), printed.

```python
r = R[R.fit == 'ridge'].sort_values('inc')  # the ridge, increasing inclination (needed by np.interp)
ki = motion_incs()  # {label: (median, p16, p84)} inclination [deg] from the motions
for k, (m, lo, hi) in ki.items():
    # Ridge q at the median, 16th and 84th percentile inclinations (linear interpolation between grid points).
    q = np.interp([m, lo, hi], r.inc, r.q)
    print(f'inc from motions ({k}) {m:.1f} ({lo:.1f}-{hi:.1f}) -> q {q[0]:.3f} ({min(q[1:]):.3f}-{max(q[1:]):.3f})')
```

#### Robustness of q at the giants' tilt
With the inclination and line of nodes of the metal-poor giants: the fit to the data; two refits with a coherent modulus error
that grows linearly with the sky offset across the line of nodes to $\pm$`DIPOLE` mag at the outer edge of the cells; and
`NBOOT` bootstrap resamples of the stars. Written to `results/shape_rrl_depth.csv`.

```python
# q at the kinematic inclination of the metal-poor giants: bootstrap of the stars, and a coherent modulus gradient across their
# line of nodes (DIPOLE at the footprint edge), each refitted with the inclination held.
inc, theta = ki['metal-poor giants'][0], kin_inc('kin_mcmc_data_ellipsoid', OLD, 'theta')[0]  # deg
# Fit to the data with the inclination held at the giants' value; xs[2] is q, xs[1] theta [deg] (shape.CELL_PARAMS order).
xs = shape.fit_cells(stats, fixed={'inc': inc}, start=best)[0]
rows = [dict(case='data', q=xs[2], theta=xs[1])]
from lmcold import geometry as geo
from lmcold.constants import LMC
r = shape.rhat(t.ra.to_numpy(), t.dec.to_numpy())  # unit vectors (N, 3) of the stars
r0, eE, eN = (np.asarray(v) for v in geo.sky_basis(np.deg2rad(LMC['ra0']), np.deg2rad(LMC['dec0'])))  # centre, east, north
across = shape.rho(r) * np.sin(np.arctan2(r @ eE, r @ eN) - np.deg2rad(theta))  # deg, sky offset across the line of nodes
for sign in (1, -1):
    # moduli shifted by sign * DIPOLE * (offset across the nodes / outer cell radius CELL_RHO[-1] = 7 deg)
    st = shape.cell_stats(t.ra.to_numpy(), t.dec.to_numpy(), t.dmu.to_numpy() + sign * DIPOLE * across / shape.CELL_RHO[-1])
    x = shape.fit_cells(st, fixed={'inc': inc}, start=xs)[0]
    rows.append(dict(case=f'dipole {sign * DIPOLE:+.2f}', q=x[2], theta=x[1]))
rng = np.random.default_rng(3)  # fixed seed of the bootstrap
for b in range(NBOOT):
    k = rng.integers(0, len(t), len(t))  # indices of one bootstrap resample of the stars
    x = shape.fit_cells(shape.cell_stats(t.ra.to_numpy()[k], t.dec.to_numpy()[k], t.dmu.to_numpy()[k]), fixed={'inc': inc}, start=xs)[0]
    rows.append(dict(case='bootstrap', q=x[2], theta=x[1]))
# Write the cases and print q with its bootstrap spread and the two dipole refits.
D = pd.DataFrame(rows)
D.to_csv(ROOT / 'results/shape_rrl_depth.csv', index=False, float_format='%.4f')
bq = D[D.case == 'bootstrap'].q
print(f'q at i = {inc:.1f}: {xs[2]:.3f}; bootstrap sd {bq.std():.3f} (16-84: {bq.quantile(0.16):.3f}-{bq.quantile(0.84):.3f}); '
      + ', '.join(f'{c} -> {q:.3f}' for c, q in D[D.case.str.startswith('dipole')][['case', 'q']].to_numpy()))
```

#### Comparison with the motions
The axis ratio the motions of the metal-poor giants require (three potentials) and the free fit, printed side by side.

```python
v = virial_q()  # rotation-implied q of the OLD bins, per potential
print('virial q:', v[['population', 'potential', 'q', 'q_p16', 'q_p84']].round(3).to_string(index=False))
print('free fit:', dict(zip(shape.CELL_PARAMS, np.round(best, 3))))
```

### Elongated mocks

`scripts/shape_rrl_triaxial.py` tests whether an elongation in the plane explains the free fit. It draws mock RR Lyrae, as many as
the data have in the footprint, from a $K_0$ body squashed in its plane to an axis ratio $b/a$ (the in-plane short axis $b$ to the
long axis $a$) and turned by an angle from the line of nodes, places them at the geometry of the metal-poor giants, and fits their
cells with the oblate model as for the data. The oblate controls ($b/a=1$) are drawn in the same way. Its table,
`results/shape_rrl_triaxial.csv`, gives the arrow of Figure 8.2 and the mocks of the checks below.

<details><summary>About <code>scripts/shape_rrl_triaxial.py</code></summary>

```text
Does an in-plane elongation explain the oblate sky-cell fit of the RR Lyrae? (paper Section 5, Appendix; KB chapter 'The shape of
the old population', the arrow of Figure paper_shape_rrl and the mocks of section 'The RR Lyrae cells')

Mock RR Lyrae (lmcold.shape.mock, N stars as in the data footprint, noise floor 0.07 mag, no outliers) of a triaxial K0 body: an
oblate K0 spheroid (scale length H, vertical axis ratio c of C_ALL) squashed along one in-plane axis to BA, that axis turned by PHI from the line
of nodes, placed at the old population's kinematic geometry (inclination and line of nodes of the metal-poor giants, stage-1
ellipsoid fits, results/kin_mcmc_data_ellipsoid.csv). Axis ratios as the RR Lyrae ellipsoid of Cusano et al. (2021, MNRAS 504, 1,
https://doi.org/10.1093/mnras/stab901: 6.5 : 4.6 : 3.7 kpc, b/a 0.71, c/a 0.57) and a milder b/a, with their c/a and the thinner c/a
that the RR Lyrae depth gradient gives at the kinematic inclination (0.41); BA = 1 is the oblate control.
Each mock is fitted with the oblate model of lmcold.shape.fit_cells (all five parameters free), as the data.

Reads: data/rrab.parquet (Zenodo bundle 1; only the number of RR Lyrae in the footprint), results/kin_mcmc_data_ellipsoid.csv
(scripts/fit_kinematics_mcmc.py) and results/shape_rrl_cells.csv (scripts/shape_rrl_cells.py, the data's free fit).
Writes results/shape_rrl_triaxial.csv (c, ba, phi, fitted inc, theta, q, h, delta, q_sky, g; and the data's free fit, last row with
c, ba, phi empty), read by scripts/paper_shape_rrl.py and scripts/paper_text_shape.py.

Run: .venv/bin/python scripts/shape_rrl_triaxial.py   (after scripts/shape_rrl_cells.py)
```
</details>

*Shown, not run here: results/shape_rrl_cells.csv (scripts/shape_rrl_cells.py); oblate fits of mock RR Lyrae catalogues of the size of the data. The cells below are the script itself; run it from the repository with* `python scripts/shape_rrl_triaxial.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The scale length and vertical axis ratios of the mock bodies, and the cases: the oblate control and two in-plane axis ratios $b/a$,
each with its major axis at six angles from the line of nodes.

```python
import numpy as np
import pandas as pd

from lmcold import shape
from lmcold.spheroid import spheroid
from lmcold.sources import ROOT
from shape_rrl_cells import OLD, kin_inc

H = 1.7  # kpc, scale length of the mock K0 bodies
C_ALL = (0.57, 0.41)  # vertical / major axis ratios c/a: Cusano et al. (2021), and q of the RR Lyrae depth gradient at the giants' tilt
CASES = [(1.0, 0.0)] + [(ba, phi) for ba in (0.71, 0.85) for phi in (0, 30, 60, 90, 120, 150)]  # (b/a, PHI [deg]); b/a = 1: oblate control
```

#### Load the data
The geometry of the metal-poor giants from their motions (inclination and line of nodes, deg), and the number of RR Lyrae in the
footprint, which each mock reproduces. The data's free fit is read at the end from `results/shape_rrl_cells.csv`.

```python
rng = np.random.default_rng(17)   # fixed seed of all mocks
inc, theta = (kin_inc('kin_mcmc_data_ellipsoid', OLD, k)[0] for k in ('inc', 'theta'))  # deg
# RRab members with a distance modulus from the OGLE I-band Wesenheit (dmu_src 'I').
d = pd.read_parquet(ROOT / 'data/rrab.parquet')
d = d[d.member & (d.dmu_src == 'I')]
n = int(shape.in_footprint(d.ra.to_numpy(), d.dec.to_numpy(), d.dmu.to_numpy()).sum())  # RR Lyrae in the footprint
```

#### Mocks fitted as oblate
For each $c/a$ one oblate K0 spheroid is drawn; for each case it is squashed along its in-plane $y$ axis to $b/a$, turned by the
angle $\phi$ of the major axis from the line of nodes, observed at the giants' geometry (`lmcold.shape.mock`), counted in the sky
cells and fitted with the oblate model. The fitted $(i,q)$ is also expressed as the sky outline $q_{\rm sky}$ and depth gradient $g$.

```python
rows = []
for cz in C_ALL:
    # shape.PARAMS vector: inc, theta [deg], q, h [kpc], delta = 0 mag, outlier fraction f = 0, noise floor 0.07 mag in each annulus
    p = np.array([inc, theta, cz, H, 0.0, 0.0] + [0.07] * 6)
    base = spheroid(H, cz, 40 * n, rng)  # disc-frame positions (40 n, 3) [kpc], the pool from which shape.mock places the stars
    for ba, phi in CASES:
        c, s = np.cos(np.deg2rad(phi)), np.sin(np.deg2rad(phi))
        X = base * [1, ba, 1] @ np.array([[c, s, 0], [-s, c, 0], [0, 0, 1]])  # squashed along y, then turned by phi
        # Mock stars from the pool X at the giants' geometry, binned in sky cells, fitted as oblate: x = CELL_PARAMS values.
        x, ll = shape.fit_cells(shape.cell_stats(*shape.mock(p, n, rng, X=X)))
        rows.append(dict(c=cz, ba=ba, phi=phi, **dict(zip(shape.CELL_PARAMS, x)),
                         **dict(zip(('q_sky', 'g'), shape.projected(x[0], x[2])))))
        print({k: round(float(v), 3) for k, v in rows[-1].items()}, flush=True)
```

#### The data's fit, for comparison
The free oblate fit of the RR Lyrae cells is appended as the last row, with its $q_{\rm sky}$ and $g$.

```python
# Free oblate fit of the data (scripts/shape_rrl_cells.py), appended with c, b/a, phi NaN.
R = pd.read_csv(ROOT / 'results/shape_rrl_cells.csv')
f = R[R.fit == 'free'].iloc[0]
rows.append(dict(c=np.nan, ba=np.nan, phi=np.nan, **{k: f[k] for k in shape.CELL_PARAMS}, **dict(zip(('q_sky', 'g'), shape.projected(f.inc, f.q)))))
print(f'truth: inc {inc:.1f} theta {theta:.1f}; n {n}')
# One row per mock (c, b/a, phi, fitted CELL_PARAMS, q_sky, g) and the data row last.
pd.DataFrame(rows).to_csv(ROOT / 'results/shape_rrl_triaxial.csv', index=False, float_format='%.4f')
```

### The inclination and flattening of the old population

<details><summary>About <code>scripts/paper_shape_rrl.py</code></summary>

```text
Paper figure (Section 5): the inclination-flattening plane of the old population, from its RR Lyrae density and from its motions.

Curves (lmcold.shape.projected, at the values of the free sky-cell fit, results/shape_rrl_cells.csv): constant outline axis ratio
on the sky q_sky (counts; blue) and constant near-to-far depth gradient g (median moduli; red) of an inclined oblate body. They
cross at the free fit (black dot: counts + distances alone). Vertical bands: the inclination from the motions (scripts/
shape_rrl_cells.motion_incs: the metal-poor giants, darker; the RR Lyrae proper motions, lighter); open circle: q of the ridge
fit (inclination held) at the giants' inclination. Dashed lines: the rotation-implied q of the metal-poor giant bins (lowest 16th and
highest 84th percentile over P1-P3, results/shape_virial.csv). Arrow: an in-plane elongated mock at the giants' geometry
(results/shape_rrl_triaxial.csv, ARROW: c/a, b/a, angle of the major axis from the nodes), from its true (inclination, c/a) to
the oblate fit of its cells. The model fits the cells poorly (chi^2 per cell ~3-6, scripts/paper_shape_rrl_residuals.py), so no
likelihood contours are drawn.

Writes figures/paper_shape_rrl.png (KB chapter 'The shape of the old population', Figure paper_shape_rrl).

Run: .venv/bin/python scripts/paper_shape_rrl.py   (after scripts/shape_rrl_cells.py, shape_rrl_triaxial.py, shape_virial.py)
```
</details>

**Data read by `scripts/paper_shape_rrl.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |
| `results/shape_rrl_cells.csv` | repository | oblate-model fits of the RR Lyrae sky cells; written by `python scripts/shape_rrl_cells.py` |
| `results/shape_rrl_triaxial.csv` | repository | oblate fits of mocks elongated in their plane; written by `python scripts/shape_rrl_triaxial.py` |
| `results/shape_virial.csv` | repository | axis ratio implied by the motions (tensor virial theorem) per giant bin and potential; written by `python scripts/shape_virial.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Plot limits, the colours of the two curves, and which elongated mock the arrow shows.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lmcold import shape
from lmcold.layout import panel_grid
from lmcold.sources import ROOT
from shape_rrl_cells import motion_incs, virial_q

XLIM, YLIM = (10.0, 70.0), (0.0, 1.0)  # deg, q
SKY, DEPTH = '#2166ac', '#b2182b'  # colours of the two curves
ARROW = (0.41, 0.85, 150.0)  # mock: c/a, b/a, major-axis angle [deg]

#### Load the data
The fits of the RR Lyrae sky cells from `results/shape_rrl_cells.csv` (`scripts/shape_rrl_cells.py`); the tilts from the motions
through `motion_incs()` (posterior summaries in `results/kin_mcmc_data_ellipsoid.csv` and
`results/kin_mcmc_data_ellipsoid_outer.csv`) and the axis ratios the motions require through `virial_q()`
(`results/shape_virial.csv`), both of `scripts/shape_rrl_cells.py`; the elongated mocks from `results/shape_rrl_triaxial.csv`
(`scripts/shape_rrl_triaxial.py`).

In [ ]:
R = pd.read_csv(ROOT / 'results/shape_rrl_cells.csv')  # free oblate fit and the ridge of fits at fixed inclination
ki = motion_incs()  # {label: (median, p16, p84)} inclination [deg] from the motions
v = virial_q()  # q required by the rotation of the metal-poor giant bins, all potentials, no central floor
T = pd.read_csv(ROOT / 'results/shape_rrl_triaxial.csv')  # oblate fits of elongated mocks (c, ba, phi)

#### The fits of the RR Lyrae sky cells
The free oblate fit of the counts and median distances in the sky cells, the 'ridge' of fits with the inclination held at a grid
of values, and the elongated mock with the axis ratios and angle `ARROW`.

In [ ]:
free = R[R.fit == 'free'].iloc[0]            # free oblate fit: inc [deg], q
r = R[R.fit == 'ridge'].sort_values('inc')   # best q at each held inclination, sorted by inclination
# The mock with c/a, b/a and angle equal to ARROW, and its oblate fit (inc, q).
t = T[np.isclose(T.c, ARROW[0]) & np.isclose(T.ba, ARROW[1]) & np.isclose(T.phi, ARROW[2])].iloc[0]

#### The figure
The two curves are the $(i,q)$ of an inclined oblate body with the observed sky outline $q_{\rm sky}$ and depth gradient $g$
(`lmcold.shape.projected`); grey bands are the tilts from the motions; the open circle is $q$ from the depth gradient at the
giants' tilt; the arrow goes from the elongated mock's true $(i, c/a)$ to its oblate fit.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
fig, ax = panel_grid(1, 1, width=3.6, left=0.55, right=0.15, top=0.25)
a = ax[0, 0]   # the single panel
I, Q = np.meshgrid(np.linspace(*XLIM, 601), np.linspace(*YLIM, 401))  # inclination [deg] and q grid for the two contours
for k, colour, text, xy in ((0, SKY, 'outline on the sky\n$q_{\\rm sky}=%.2f$', (62, 0.55)),
                            (1, DEPTH, 'near-to-far distance\ngradient $g=%.2f$', (52, 0.85))):
    # Observed q_sky (k = 0) or depth gradient g (k = 1) of the free fit; its contour over (i, q); label at xy.
    z0 = shape.projected(free.inc, free.q)[k]
    a.contour(I, Q, shape.projected(I, Q)[k], levels=[z0], colors=[colour], linewidths=1.5)
    a.text(*xy, text % z0, color=colour, fontsize=7, ha='center', va='center')
# Grey bands: 16th-84th percentiles of the inclination from the motions (first entry darker, label at its median).
for k, (name, (m, lo, hi)) in enumerate(ki.items()):
    a.axvspan(lo, hi, color='k', alpha=0.15 if k == 0 else 0.06, lw=0)
    a.text(m if k == 0 else lo - 0.5, 0.97 - 0.09 * k, 'tilt from motions:\n' + name, fontsize=6, va='top',
           ha='left' if k == 0 else 'right')
# Open circle: q of the ridge interpolated at the median tilt [deg] of the metal-poor giants.
m = ki['metal-poor giants'][0]
qk = np.interp(m, r.inc, r.q)
a.plot(m, qk, 'o', mfc='w', mec='k', ms=5, zorder=5)
a.annotate(f'$q={qk:.2f}$', (m, qk), xytext=(5, -10), textcoords='offset points', fontsize=7)
# Dashed lines: lowest 16th and highest 84th percentile over results/shape_virial.csv of the q the rotation requires.
for y in (v.q_p16.min(), v.q_p84.max()):
    a.axhline(y, color='k', ls='--', lw=0.8)
a.text(XLIM[1] - 1, v.q_p84.max() + 0.01, 'q needed by the rotation (motions)', fontsize=6, ha='right', va='bottom')
# Arrow from the mock's true (giants' tilt, c/a) to its oblate fit (inc, q); the text is placed at (44, 0.3).
a.annotate('', (t.inc, t.q), (m, ARROW[0]), arrowprops=dict(arrowstyle='->', color='0.4', lw=0.8, shrinkA=3, shrinkB=3))
a.text(44, 0.3, f'elongated mock at the\nmotions tilt\n($b/a={ARROW[1]}$, $c/a={ARROW[0]}$)\nfitted as oblate', fontsize=6, color='0.4', ha='center')
# Filled circle: the free fit of counts and distances.
a.plot(free.inc, free.q, 'ko', ms=5, zorder=5)
a.annotate(f'counts + distances alone\n$i={free.inc:.0f}^\\circ$, $q={free.q:.2f}$', (free.inc, free.q), xytext=(-95, 12),
           textcoords='offset points', fontsize=7)
a.set(xlabel='inclination $i$ [deg]', ylabel='axis ratio $q$ (0: thin disc, 1: sphere)', xlim=XLIM, ylim=YLIM)
fig.savefig(ROOT / 'figures/paper_shape_rrl.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_shape_rrl.png')

**Figure 8.2.** Inclination $i$ (deg) and axis ratio $q$ of the old population. Curves: the $(i,q)$ of an inclined oblate body with the RR Lyrae's sky outline
($q_{\rm sky}$ = 0.84, blue) and near-to-far depth gradient ($g$ = 0.36, red); black dot: their crossing. Grey bands: the inclination
from the motions of the metal-poor giants (dark) and of the RR Lyrae proper motions (light); open circle: $q$ from the depth gradient
at the giants' inclination. Dashed: $q$ implied by the motions of the metal-poor giants (16th–84th percentiles over P1–P3). Arrow: a
mock with an in-plane elongation ($b/a$ = 0.85, $c/a$ = 0.41) at the giants' geometry, from its true $(i,c/a)$ to its oblate
fit. If the population were round in its plane, its shape would lie where the curves cross. At the inclination of the motions,
however, no such body is as oval on the sky as the RR Lyrae (the blue curve does not reach it), so the crossing measures their
elongation in the plane, as the arrow shows, not their tilt. What survives is the red curve: at the motions' tilt the depth
gradient gives $q\approx$ 0.41, consistent with the flattening the motions require. `scripts/paper_shape_rrl.py`

## The size of each population

The kinematic modelling also measures how far each population extends. For each giant [Fe/H] bin, `scripts/kin_vc_drift.py`
(chapter 7) counts the BOSS giants in rings of in-plane radius $R$, each star weighted for the selection of the spectra, the sky
coverage and the chance that it belongs to the Milky Way, and fits an exponential surface density $\Sigma(R)\propto e^{-R/h}$,
with scale length $h$, between 2 and 8 kpc; inside 2 kpc the bar
dominates. The RR Lyrae are counted the same way. The scale length grows from 0.9 kpc for the most metal-rich giants
to 1.9 kpc for the most metal-poor (1.7 kpc for the RR Lyrae): the oldest stars are the most extended
(Figure 8.3). The half-mass radius of an exponential disc is $R_e = 1.68\,h$, which gives 1.5,
3.2 and 2.9 kpc. These values assume that the exponential continues inside
2 kpc, which the data do not support for every bin: the surface density of the most metal-poor giants there
falls below its exponential, and that of the most metal-rich giants falls faster than an exponential at large radii. Whether the
inner deficit is real or a result of the selection in the crowded bar region we have not tested.

All these are present-day sizes. Heating and radial migration since the stars formed, and the LMC's interactions with the SMC and
the Milky Way, may have made the old disc larger than it was at birth.

<details><summary>About <code>scripts/fig_sizes.py</code></summary>

```text
Sizes of the LMC's populations: the selection-weighted surface density of the giants in each [Fe/H] bin and of the RR Lyrae
against in-plane radius, with the exponential scale lengths h that scripts/kin_vc_drift.py fits to them (results/kin_vc_drift_h.csv).

The surface densities are made exactly as in kin_vc_drift.py (its functions and settings are imported): BOSS giants of the mass
sample weighted by selection weight x coverage fill x (1 - Milky Way fraction), RR Lyrae members with weight 1, in 1-kpc annuli of
the in-plane radius R of the joint ellipsoid fit; h is the exponential fitted over H_FIT (2-8 kpc; inside 2 kpc the bar). The
half-mass radius of a projected exponential disc is R_e = RE_PER_H h; quoting it extrapolates the exponential inside 2 kpc.
These are present-day sizes: heating and radial migration since the stars formed may have changed them.

Reads the BOSS mass sample (lmcold.mass_samples), data/giant_foreground.parquet, data/rrab.parquet, results/kin_mcmc_data_ellipsoid.csv
and results/kin_vc_drift_h.csv. Writes figures/fig_sizes.png: (a) surface density (each normalised to its fitted exponential at
R = 4 kpc) against R, the fits as lines over H_FIT, the region inside H_FIT[0] shaded; (b) h (filled) and R_e (open) against the
median [Fe/H] of each giant bin, the RR Lyrae at their median photometric [Fe/H]. Diagnostic figure (KB, Shape chapter).

Run: .venv/bin/python scripts/fig_sizes.py   (about a minute)
```
</details>

**Data read by `scripts/fig_sizes.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/giant_foreground.parquet` | Zenodo bundle 1 | Milky Way foreground model per giant, from the mass analysis |
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_vc_drift_h.csv` | repository | fixed input |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

### Settings
The annuli, fit range and radius function are imported from `scripts/kin_vc_drift.py`, so that the profiles are the ones its scale
lengths were fitted to.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import numpy as np
import pandas as pd

from kin_vc_drift import H_FIT, R_EDGES, joint_params, radius
from lmcold import samples
from lmcold.kinfit import R_SIG
from lmcold.layout import panel_grid
from lmcold.mass_samples import samples as mass_samples
from lmcold.sources import DATA, ROOT

RE_PER_H = 1.678   # half-mass radius / scale length of a projected exponential disc
R_NORM = R_SIG     # kpc, where each profile is normalised (the radius of the kinematic quantities)

### Load the data
The joint-fit disc frame, the BOSS mass sample with its weights, the RR Lyrae members and the fitted scale lengths.

In [ ]:
# Disc frame: stage-2 posterior medians of the joint ellipsoid fit (shared parameters plus those of the first
# giant bin); only the centre, inclination and position angle enter radius()
frame = joint_params()['giants [Fe/H] < -1.5']
# BOSS mass sample (first of the three returned tables), with the coverage fill and Milky Way fraction per star
b, _, _ = mass_samples()
fg = pd.read_parquet(DATA / 'giant_foreground.parquet')
b = b.merge(fg[['source_id', 'fill', 'f_mw']], on='source_id', validate='one_to_one')
# W: total weight = selection weight x coverage fill x (1 - Milky Way fraction); R: in-plane radius [kpc]
b['W'] = b.w * b['fill'] * (1 - b.f_mw)
b['R'] = radius(frame, b.ra.to_numpy(), b.dec.to_numpy())
# RR Lyrae members; H: fitted scale lengths h [kpc] with percentiles, indexed by population
rr = pd.read_parquet(DATA / 'rrab.parquet')
rr = rr[rr.member]
H = pd.read_csv(ROOT / 'results/kin_vc_drift_h.csv').set_index('population')

### Surface densities
For each population the weighted counts per annulus over the annulus area, normalised to its fitted exponential at R_NORM.

In [ ]:
# Annulus centres [kpc] and areas [kpc^2]; fit_range: annuli whose centre lies inside H_FIT
mid, area = .5 * (R_EDGES[1:] + R_EDGES[:-1]), np.pi * np.diff(R_EDGES ** 2)
fit_range = (mid > H_FIT[0]) & (mid < H_FIT[1])
# (name, R [kpc], weights, median [Fe/H]) of each giant [Fe/H] bin of lmcold.samples, then the RR Lyrae
pops = [(f'giants {lab}', b[(b.feh >= lo) & (b.feh < hi)].R.to_numpy(), b[(b.feh >= lo) & (b.feh < hi)].W.to_numpy(),
         np.median(b[(b.feh >= lo) & (b.feh < hi)].feh)) for lab, (lo, hi) in zip([lab for lab, _ in samples.giants()], samples.FEH_BINS)]
pops.append(('RR Lyrae', radius(frame, rr.ra.to_numpy(), rr.dec.to_numpy()), np.ones(len(rr)), np.nanmedian(rr.feh)))   # 4 per cent have no [Fe/H]
prof = {}
for name, R, w, feh in pops:
    sigma = np.histogram(R, R_EDGES, weights=w)[0] / area      # weighted stars per kpc^2
    # Fitted scale length [kpc]; k: annuli in the fit range with stars
    h = H.loc[name, 'h']
    k = fit_range & (sigma > 0)
    a0 = np.mean(np.log(sigma[k]) + mid[k] / h)                # intercept of ln Sigma = a0 - R / h at the fitted slope
    prof[name] = (sigma / np.exp(a0 - R_NORM / h), h, feh)

### Figure
(a) The normalised profiles and their exponentials; (b) the scale lengths and the extrapolated half-mass radii against [Fe/H].

In [ ]:
# Two equal panels: a = profiles, c = sizes; one colour per population (blues metal-poor, oranges metal-rich)
fig, ax = panel_grid(1, 2, width=7.0, left=0.65, top=0.6, hgap=0.7)
a, c = ax[0, 0], ax[0, 1]
colours = dict(zip(prof, ['#08306b', '#2171b5', '#6baed6', '#fd8d3c', '#d94801', 'k']))
# Radii [kpc] at which the fitted exponentials are drawn
x = np.linspace(*H_FIT, 50)
for name, (s, h, feh) in prof.items():
    # Points: measured profile (non-empty annuli); line: exp(-(R - R_NORM)/h), equal to 1 at R_NORM
    ls = '--' if name == 'RR Lyrae' else '-'
    a.semilogy(mid[s > 0], s[s > 0], 'o' if ls == '-' else 's', color=colours[name], ms=3)
    a.semilogy(x, np.exp(-(x - R_NORM) / h), ls, color=colours[name], lw=1, label=f'{name.replace("giants ", "")}: h = {h:.1f} kpc')
# Shade the region inside H_FIT[0] (not fitted); legend above the panel
a.axvspan(0, H_FIT[0], color='0.9', zorder=0)
a.set(xlabel='in-plane radius R [kpc]', ylabel=rf'$\Sigma(R)/\Sigma_{{\rm exp}}({R_NORM:g}\,{{\rm kpc}})$', xlim=(0, R_EDGES[-1]))
a.legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=6, frameon=False, ncol=2)
for name, (s, h, feh) in prof.items():
    # Filled: h with its 16th-84th percentiles; open: R_e = RE_PER_H h
    lo, hi = H.loc[name, 'h_p16'], H.loc[name, 'h_p84']
    mk = 's' if name == 'RR Lyrae' else 'o'
    c.errorbar(feh, h, [[h - lo], [hi - h]], fmt=mk, color=colours[name])
    c.plot(feh, RE_PER_H * h, mk, mfc='none', color=colours[name])
# Legend entries only (empty data), placed above the panel
c.plot([], [], 'ko', label='scale length h (fitted, 2-8 kpc)')
c.plot([], [], 'ko', mfc='none', label=r'$R_e=1.68\,h$ (extrapolated inside 2 kpc)')
c.set(xlabel='median [Fe/H]', ylabel='size [kpc]', ylim=(0, None))
c.legend(loc='lower left', bbox_to_anchor=(0, 1.02), fontsize=6, frameon=False)
fig.savefig(ROOT / 'figures/fig_sizes.png', dpi=200)

In [ ]:
from IPython.display import Image
Image(filename='figures/fig_sizes.png')

**Figure 8.3.** (a) Surface density of each giant [Fe/H] bin (circles) and of the RR Lyrae (black squares) against the
in-plane radius $R$, each normalised to its fitted exponential at $R$ = 4 kpc; lines: the exponentials fitted between
2 and 8 kpc, with their scale lengths in the key; grey: the bar region, not fitted.
(b) The scale lengths (filled, with 16th–84th bootstrap percentiles) and the half-mass radii $R_e=1.68\,h$ they imply if the
exponential continued inwards (open), against the median [Fe/H] of each sample.

## The shape and the kinematic model

A spheroid with $q\approx0.5$ and $h$ = 1.9 kpc has a vertical extent comparable to the radii we fit: the rms height of its stars
is $q\sqrt3\,h\approx$ 1.7 kpc, thicker than the 1-kpc disc of the tests of the kinematic model. We therefore drew a mock
with this shape ('hot spheroid': $q$ = 0.5 with the 1.5-kpc scale length of the other mocks, rms height 1.3 kpc;
radial, azimuthal and vertical velocity dispersions
$(\sigma_R,\sigma_\phi,\sigma_z)$ = (40, 28, 30) km/s, asymmetric drift, inclination 28°). With proper motions and
line-of-sight velocities the thin-disc model recovers its dispersions to within 0.6 km/s and its inclination to 0.3°
(line of nodes 3° low, as for the 1-kpc disc). The thickness of the old population therefore does not bias its
kinematics, and the lower inclination of the metal-poor giants (25° against 28°) is not a projection of their
thickness. With proper motions alone the same mock gives $\sigma_z$ = 50 ± 8 km/s for 30 and an inclination of
17°: the proper-motion inclination of the RR Lyrae (22.3°) is not a reliable tilt for a thick population, and we use that
of the metal-poor giants.

## Summary of the shape

1. To support their rotation and dispersions in a steady state, the oldest, most metal-poor stars ([Fe/H] < -1.1) must form a
   thick, rotating disc with $q\approx0.5$ (0.46–0.57 over the three potentials); the flattening grows with metallicity, to
   $q$ = 0.25–0.38 above [Fe/H] = -0.5. The procedure recovers the shapes of simulated analogues to -0.02 in their own
   potential; the uncertainty is the LMC's potential.
2. The RR Lyrae, the same population seen through distances, agree: at the tilt of the motions their near-to-far distance gradient
   gives $q\approx$ 0.41 (0.19–0.54 with plausible distance errors).
3. On their own the RR Lyrae suggest a tilt of 48°, but this is an illusion: their oval outline on the sky comes from an
   elongation of the population in its plane, which a round-in-plane model mistakes for tilt.
4. The thin-disc kinematic model describes a population this thick without bias when line-of-sight velocities are available;
   proper motions alone underestimate its tilt.

## Checks

### The flattening of simulated analogues

The Auriga and FIRE-2 zooms contain all particle types, so their potential is known. For each, we compute the axisymmetrised
potential of all particles within 50 kpc in the disc frame of the stars (200 000 particles drawn by mass at random azimuths, softened
gravity summed on a grid in $R$ and $|z|$; on a [Hernquist (1990)](https://doi.org/10.1086/168845) sphere this reproduces the
analytic $W_{xx}/W_{zz}$ of a tracer with $q$ = 1 to 1 per cent, `tests/test_virial.py`). For the two metal-poor bins we take the
kinematic model fitted to the analogue's mock giants, realise it on a $K_0$ spheroid with the bin's scale length, and solve the
equation in that potential, exactly as for the LMC. The true axis ratio is that of the bin's particles,
$q=(2\langle z^2\rangle/\langle R^2\rangle)^{1/2}$ (angle brackets: mean over the bin's star particles) at $R$ < 10 kpc, $|z|$ < 5 kpc. Over the 16 bins (true $q$ = 0.49–0.60)
the recovered $q$ is low by -0.02 (16th–84th percentiles -0.04 to 0.00; range -0.08 to +0.04). In the fixed
potentials P1, P2 and P3 the offsets are +0.13 (+0.07 to +0.15), +0.03 (-0.02 to +0.04) and
+0.08 (+0.03 to +0.10). We do not test the particles themselves against the theorem: their metallicities are
mapped on the LMC scale only inside the cone of the observed footprint, so a bin is a truncated tracer.

#### The test script

`scripts/shape_virial_sims.py` runs this test. For each Auriga and FIRE-2 analogue it computes the potential of all particles in the
disc frame of the stars; for each of the two metal-poor bins it measures the true axis ratio and the scale length of the bin's
particles, weighted by the giants they would contribute to the BOSS sample, and solves for $q$ with the kinematic model fitted to
the analogue's mock giants, in the analogue's own potential and in P1, P2 and P3. Its table, `results/shape_virial_sims.csv`, gives
the numbers of this check.

<details><summary>About <code>scripts/shape_virial_sims.py</code></summary>

```text
Test of the rotation-implied flattening (lmcold.virial, paper Section 5) on simulated analogues, each in its own potential (KB
chapter 'The shape of the old population', section 'The flattening of simulated analogues').

For every zoom analogue with all particle types (Auriga L3 x 5, FIRE-2 x 3; lmcold.sims.masses; the TNG50 cutouts hold stars only)
and the two metal-poor [Fe/H] bins (populations 0 and 1: [Fe/H] < -1.5 and -1.5 to -1.1 on the LMC scale, lmcold.samples.FEH_BINS):
- q_true, h: the axis ratio (lmcold.spheroid.shape) and exponential scale length (scripts/kin_vc_drift.scale_length: the weighted
  surface density in 1-kpc annuli over 2-8 kpc, as for the LMC) of the bin's particles (scripts/sims_observe.particles, weighted by
  their expected BOSS giants E);
- potential: lmcold.virial.grid_potential of all particles in the disc frame of the stars (lmcold.sims.disc_frame);
- q_model: lmcold.virial.flattening of the kinematic model fitted to the analogue's mock giants (results/sims_observed.csv, p_*),
  in the analogue's potential ('sim') and in P1-P3: what the LMC procedure gives.

Reads: the particles of the Auriga and FIRE-2 analogues, stars (lmcold.sims.load) and all particle types (lmcold.sims.masses), from
the snapshots fetched by scripts/fetch_auriga.py and scripts/fetch_fire2.py (docs/release.md: about 120 GB and 8 GB); the inputs of scripts/sims_observe.setup, and results/sims_observed.csv (the kinematic
fits to the analogues' mock giants, written by scripts/sims_observe.py).
Writes results/shape_virial_sims.csv (suite, name, population, q_true, h, q_model_sim, q_model_P1, q_model_P2, q_model_P3), read by
scripts/paper_text_shape.py.
The particles themselves are not tested against the theorem: their LMC-scale [Fe/H] exists only inside the RHO_MASS cone of
scripts/sims_observe, so the bins are truncated tracers.

Run: .venv/bin/python scripts/shape_virial_sims.py   (after scripts/sims_observe.py)
```
</details>

*Shown, not run here: the Auriga and FIRE-2 snapshots with all particle types (scripts/fetch_auriga.py, scripts/fetch_fire2.py; about 120 GB and 8 GB) and results/sims_observed.csv from scripts/sims_observe.py (about 13 h). The cells below are the script itself; run it from the repository with* `python scripts/shape_virial_sims.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

##### Settings
The modules: the particle loading and observing steps of `scripts/sims_observe.py`, the scale-length fit of `scripts/kin_vc_drift.py`,
and the virial solver of `lmcold.virial`.

```python
import numpy as np
import pandas as pd

from kin_vc_drift import scale_length
from sims_observe import particles, setup
from lmcold import kinfit, samples, sims, spheroid, virial
from lmcold.sources import ROOT

P = kinfit.PARAMS  # parameter names of a kinematic model vector (columns p_<name> of results/sims_observed.csv)
```

##### Load the data
The inputs shared by all analogues (`sims_observe.setup`: the LMC frame, the cumulative-mass knots of the [Fe/H] mapping, the
$n_{\rm box}$ and survival grids) and the kinematic fits to each analogue's mock giants. The particles are read per analogue in
the next step.

```python
rng = np.random.default_rng(13)
frame, _, _, knots, nbox, survival = setup()
obs = pd.read_csv(ROOT / 'results/sims_observed.csv', dtype={'name': str})  # fitted model p_* per suite, name, population
```

##### True and recovered axis ratios
For each Auriga and FIRE-2 analogue: the axisymmetrised potential of all its particles in the disc frame of its stars; then, for
the two metal-poor bins, the true axis ratio and scale length of the bin's particles, and the axis ratio that the kinematic model
fitted to its mock giants implies in the analogue's own potential and in P1-P3.

```python
rows = []
for suite, name in [(s, n) for s, n in sims.names() if s != 'TNG50']:
    st = sims.load(suite, name, survival)  # star particles
    _, _, rot = sims.disc_frame(st['x'], st['v'], st['m'])  # rotation matrix to the disc frame of the stars
    xm, m = sims.masses(suite, name)  # all particles: positions (N, 3) [kpc], masses [Msun]
    grad = virial.grid_potential(xm @ rot.T, m, rng)  # gradient function of the analogue's potential
    X, _, _, _, feh, E, _ = particles(suite, name, frame, knots, nbox, survival)  # disc-frame X [kpc], LMC-scale [Fe/H], expected BOSS giants E
    for j in (0, 1):
        b = (feh >= samples.FEH_BINS[j][0]) & (feh < samples.FEH_BINS[j][1])
        q, h = spheroid.shape(X[b], E[b]), scale_length(np.hypot(X[b, 0], X[b, 1]), E[b], np.zeros(b.sum()))[0]  # q_true, h [kpc]
        o = obs[(obs.suite == suite) & (obs.name == name) & (obs.population == j)]
        row = dict(suite=suite, name=name, population=j, q_true=q, h=h)
        if len(o) and np.isfinite(o.vrot4.iloc[0]):  # the bin has a fitted rotation speed vrot4 (otherwise the q_model columns stay empty)
            p = o[[f'p_{n}' for n in P]].to_numpy()[0]
            row['q_model_sim'] = virial.flattening(p, h, grad, rng)[0]
            for pn, g in virial.POTENTIALS.items():
                row[f'q_model_{pn}'] = virial.flattening(p, h, g, rng)[0]
        rows.append(row)
        print({kk: (round(vv, 3) if isinstance(vv, float) else vv) for kk, vv in row.items()}, flush=True)
pd.DataFrame(rows).to_csv(ROOT / 'results/shape_virial_sims.csv', index=False, float_format='%.4f')
```

### The RR Lyrae cells

Figure 8.4 shows what the oblate model leaves in the RR Lyrae cells, for its best fit and at the inclination of the motions.
The counts have Pearson residuals from -5.4 to +8.6 in coherent patches; their $m=1$ amplitude is 0.32 at
2–2.5° and 0.45–0.58 at 5–7°, where the excess lies to the north-east (position angle
64–75°) and the deficit to the west and south-west. The medians scatter by 0.028 mag about the model against their
errors of 0.015 mag. To test whether an elongation in the plane explains the oblate fit, we drew mocks at the geometry of the
metal-poor giants with $c/a$ = 0.57 (as Cusano et al. 2021) or 0.41, squashed in the plane to $b/a$ = 0.71 or 0.85 with the
major axis at 0–150° from the line of nodes (in steps of 30°), and fitted them like the data. With the major axis within
30° of the line of nodes the oblate fits move to $i$ = 50–70° (the upper bound), $q$ = 0.67–0.78 and
$q_{\rm sky}$ = 0.72–0.84, the fitted line of nodes following the major axis; with $b/a$ = 0.85, $c/a$ = 0.41 and
the major axis at 150° they give $i$ = 50.4°, $q$ = 0.71, $\theta$ = 152°, $q_{\rm sky}$ = 0.840,
$g$ = 0.345 (data: 48.1°, 0.69, 165°, 0.844, 0.361). With the major axis at 60–120° from the line of
nodes the fits stay at $i$ = 12–21° with $q$ = 0.10–0.50, except for the most elongated and thickest mock
($b/a$ = 0.71, $c/a$ = 0.57), which goes to the upper bound at 60 and 120° as well. The oblate controls ($b/a=1$) come back at
$i$ = 19 and 22° with $q$ = 0.40 and 0.22 for a true 24.9° with 0.57 and 0.41: they slide along the curve
of constant $g$, which the counts constrain only weakly.

<details><summary>About <code>scripts/paper_shape_rrl_residuals.py</code></summary>

```text
Paper figure (Appendix): what the oblate sky-cell model leaves in the RR Lyrae counts and median moduli (paper Section 5).

The cells of lmcold.shape.cell_stats (8 rings x 24 sectors, 2-7 deg), for the free fit (counts + medians alone) and for the fit at the
old population's kinematic inclination (the ridge point at INC, results/shape_rrl_cells.csv), from lmcold.shape.cell_model:
(a, c) counts, Pearson residual (n - lam) / sqrt(lam); (b, d) median modulus minus model [mag]. Sky: east left, north up, rho [deg]
from the LMC centre; lines: the fit's line of nodes (solid) and the kinematic one of the metal-poor giants (dashed).

Writes figures/paper_shape_rrl_residuals.png (KB chapter 'The shape of the old population', Figure paper_shape_rrl_residuals) and results/shape_rrl_harmonics.csv (per ring of the free fit: m = 1 and m = 2 amplitude and
position angle of the fractional count residual n / lam - 1, and the fraction of the ring inside the OGLE-IV fields,
lmcold.ogle.in_footprint, to tell structure from coverage) and results/shape_rrl_fit_stats.csv (per fit: stars, cells, chi^2 of
the counts and of the medians, range of the count Pearson residuals, rms modulus residual and median modulus error [mag]).

Run: .venv/bin/python scripts/paper_shape_rrl_residuals.py   (after scripts/shape_rrl_cells.py)
```
</details>

**Data read by `scripts/paper_shape_rrl_residuals.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/raw/ogle4/fields-OGLE-IV-LMC.txt` | public archive | OGLE-IV field centres, LMC (LMC); Udalski et al. 2015, AcA 65, 1 |
| `data/raw/ogle4/fields-OGLE-IV-MBR.txt` | public archive | OGLE-IV field centres, MBR (Magellanic Bridge); Udalski et al. 2015, AcA 65, 1 |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/shape_rrl_cells.csv` | repository | oblate-model fits of the RR Lyrae sky cells; written by `python scripts/shape_rrl_cells.py` |

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The inclination of the second fit and the colour limits of the residual maps.

In [ ]:
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Wedge

from lmcold import ogle, shape
from lmcold.layout import panel_grid
from lmcold.sources import ROOT
from shape_rrl_cells import OLD

INC = 25.0  # deg, ridge point at the metal-poor giants' inclination (24.9; the ridge grid has 25.0)
LIM = (4.0, 0.06)  # colour limits: Pearson residual, modulus residual [mag]

#### Drawing helpers
`cells` colours the sky cells as wedges on the tangent plane; `axis` draws a line through the centre at a given position angle.

In [ ]:
def cells(a, values, norm, cmap):
    """Cell wedges coloured by values (NaN: blank); position angle from north through east, east to the left.

    a: matplotlib axes (tangent-plane coordinates xi, eta [deg]); values: one number per cell of lmcold.shape.cell_stats, ring by ring
    (len(CELL_RHO) - 1 rings x CELL_NPA sectors); norm, cmap: colour normalisation and map."""
    # angular width of one position-angle sector [deg] (360 / 24 = 15)
    w = 360 / shape.CELL_NPA
    # cells numbered ring by ring: index c -> (ring, sector)
    for c, v in enumerate(values):
        if not np.isfinite(v):
            continue
        ring, sec = divmod(c, shape.CELL_NPA)
        # inner and outer radius of the ring [deg]
        r1, r2 = shape.CELL_RHO[ring], shape.CELL_RHO[ring + 1]
        # Wedge angles run from the +xi axis; with xi = rho sin PA, eta = rho cos PA, sector PA1..PA2 spans 90 - PA2..90 - PA1
        a.add_patch(Wedge((0, 0), r2, 90 - (sec + 1) * w, 90 - sec * w, width=r2 - r1, color=cmap(norm(v)), lw=0))  # xi = rho sin PA


def axis(a, theta, **kw):
    """Line through the centre at position angle theta [deg] (east of north) on axes a, out to 7.5 deg; kw passed to plot."""
    # PA east of north -> angle from the +xi axis [rad]
    t = np.deg2rad(90 - theta)
    a.plot([-7.5 * np.cos(t), 7.5 * np.cos(t)], [-7.5 * np.sin(t), 7.5 * np.sin(t)], color='k', lw=0.8, **kw)

#### Load the data
The RR Lyrae table `data/rrab.parquet`, the fits of the sky cells from `results/shape_rrl_cells.csv`
(`scripts/shape_rrl_cells.py`) and the kinematic posterior summaries of `results/kin_mcmc_data_ellipsoid.csv`. The OGLE-IV field
centres (`data/raw/ogle4/fields-OGLE-IV-LMC.txt`, `fields-OGLE-IV-MBR.txt`) are read where they are used, inside
`lmcold.ogle.in_footprint` (ring coverage below).

In [ ]:
t = pd.read_parquet(ROOT / 'data/rrab.parquet')  # RR Lyrae with membership, distance modulus dmu and its source dmu_src
R = pd.read_csv(ROOT / 'results/shape_rrl_cells.csv')  # free oblate fit and the ridge of fits at fixed inclination
k = pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid.csv')  # ellipsoid-model posterior summaries (line of nodes theta)

#### The RR Lyrae in the sky cells
Members with an OGLE Wesenheit distance modulus, inside the 2-7 deg footprint, counted in the sky cells of `lmcold.shape.cell_stats`
(number, median modulus and its error per cell); the free fit and the fit at the kinematic inclination; the mean line of nodes of
the old population from its motions.

In [ ]:
# members whose distance modulus comes from the OGLE I-band (Wesenheit) photometry
t = t[t.member & (t.dmu_src == 'I')]
# inside the 2-7 deg annulus and the distance window (lmcold.shape.in_footprint)
t = t[shape.in_footprint(t.ra.to_numpy(), t.dec.to_numpy(), t.dmu.to_numpy())]
# per cell: star count n, median modulus med [mag] and its standard error se [mag]
n, med, se = shape.cell_stats(t.ra.to_numpy(), t.dec.to_numpy(), t.dmu.to_numpy())
# (unit vectors, cell index) of the quadrature points inside the cells
sky = shape.cell_sky()
# (label, fitted row): the free fit, and the ridge fit at inclination INC
fits = [('free fit', R[R.fit == 'free'].iloc[0]), (f'$i={INC:.0f}^\\circ$ (kinematic)',
                                                                   R[(R.fit == 'ridge') & np.isclose(R.inc, INC)].iloc[0])]
# kinematic line of nodes theta [deg]: mean of the stage-1 posterior medians of the old-population groups (OLD)
theta_kin = k[(k.stage == 1) & k.population.isin(OLD) & (k.param == 'theta')]['median'].mean()

#### Lopsidedness of the counts, ring by ring
The $m=1$ and $m=2$ Fourier amplitudes of the fractional count residual $n/\lambda-1$ of the free fit in each ring, with the
fraction of the ring inside the OGLE-IV fields (to tell structure from coverage).

In [ ]:
# parameters (inc, theta, q, h, delta) of the free fit
p = [fits[0][1][c] for c in shape.CELL_PARAMS]
# expected counts lam per cell of the free fit
lam = np.asarray(shape.cell_model(p, n, sky)[0])
# quadrature points of the whole annulus, unit vectors (M, 3)
rs = shape.footprint()[0]
# whether each point lies in an OGLE-IV field ((RA, Dec) [deg] from the unit vector)
inside = ogle.in_footprint(np.rad2deg(np.arctan2(rs[:, 1], rs[:, 0])) % 360, np.rad2deg(np.arcsin(rs[:, 2])))
# cell index of each point (-1 outside the rings)
k = shape.cell_index(rs)
# centre position angle of each sector [rad]
pa = (np.arange(shape.CELL_NPA) + 0.5) * 2 * np.pi / shape.CELL_NPA
rows = []
for ring in range(len(shape.CELL_RHO) - 1):
    # the cells of this ring, and their fractional count residual n / lam - 1
    c = slice(ring * shape.CELL_NPA, (ring + 1) * shape.CELL_NPA)
    d = n[c] / lam[c] - 1
    # coverage: fraction of the ring's quadrature points inside the OGLE-IV fields
    row = dict(rho_lo=shape.CELL_RHO[ring], rho_hi=shape.CELL_RHO[ring + 1], coverage=inside[k // shape.CELL_NPA == ring].mean())
    for m in (1, 2):
        # Fourier cosine and sine coefficients of order m; amplitude a_m and phase angle pa_m [deg] (period 360 / m)
        a, b = 2 * np.mean(d * np.cos(m * pa)), 2 * np.mean(d * np.sin(m * pa))
        row[f'a{m}'], row[f'pa{m}'] = np.hypot(a, b), np.rad2deg(np.arctan2(b, a)) / m % (360 / m)
    rows.append(row)
H = pd.DataFrame(rows)
H.to_csv(ROOT / 'results/shape_rrl_harmonics.csv', index=False, float_format='%.3f')
print(H.round(3).to_string(index=False))

#### The residual maps
Counts as Pearson residuals $(n-\lambda)/\sqrt{\lambda}$ and median moduli minus the model, for both fits; the fit statistics go to
`results/shape_rrl_fit_stats.csv`.

In [ ]:
plt.style.use(ROOT / 'src/lmcold/paper.mplstyle')
# 2 x 2 equal panels: rows = fits, columns = counts and moduli
fig, ax = panel_grid(2, 2, width=5.0, left=0.5, right=0.1, hgap=0.45, vgap=0.3, top=0.75)
# symmetric colour ranges: +-4 for the Pearson residual, +-0.06 mag for the modulus residual
norms = [Normalize(-LIM[0], LIM[0]), Normalize(-LIM[1], LIM[1])]
cmap = plt.get_cmap('RdBu_r')
stats = []
for i, (label, f) in enumerate(fits):
    # parameters of this fit; lam: expected counts; Mc: model mean modulus [mag] per cell
    p = [f[c] for c in shape.CELL_PARAMS]
    lam, Mc = (np.asarray(v) for v in shape.cell_model(p, n, sky))
    # res[0]: Pearson residual of the counts; res[1]: median modulus minus model [mag]
    res = [(n - lam) / np.sqrt(lam), med - Mc]
    print(f'{label}: counts chi2 {np.sum(res[0] ** 2):.0f}, medians chi2 {np.nansum((res[1] / se) ** 2):.0f}, '
          f'rms modulus residual {np.nanstd(res[1]):.3f} mag ({len(n)} cells)')
    # fit statistics written to results/shape_rrl_fit_stats.csv
    stats.append(dict(fit=('free', 'ridge')[i], inc=f.inc, n_stars=len(t), n_cells=len(n), chi2_counts=np.sum(res[0] ** 2),
                      chi2_medians=np.nansum((res[1] / se) ** 2), pearson_min=res[0].min(), pearson_max=res[0].max(),
                      rms_modulus=np.nanstd(res[1]), median_se=np.nanmedian(se)))
    for j in (0, 1):
        # one panel per residual: the cells, the fit's line of nodes (solid) and the kinematic one (dashed)
        a = ax[i, j]
        cells(a, res[j], norms[j], cmap)
        axis(a, f.theta)
        axis(a, theta_kin, ls='--')
        # east to the left, +-7.5 deg
        a.set(xlim=(7.5, -7.5), ylim=(-7.5, 7.5), aspect='equal')
        a.text(0.03, 0.97, f'({"abcd"[2 * i + j]}) {label}', transform=a.transAxes, fontsize=6, va='top')
    ax[i, 0].set_ylabel(r'$\eta$ [deg]')
# x labels on the bottom row; one horizontal colour bar per column, above the top panel
for j, lab in enumerate(('counts: $(n-\\lambda)/\\sqrt{\\lambda}$', 'median modulus $-$ model [mag]')):
    ax[1, j].set_xlabel(r'$\xi$ [deg] (east left)')
    pos = ax[0, j].get_position()
    cax = fig.add_axes([pos.x0, pos.y1 + 0.03, pos.width, 0.015])
    fig.colorbar(ScalarMappable(norms[j], cmap), cax=cax, orientation='horizontal').set_label(lab, fontsize=7)
    cax.xaxis.set_ticks_position('top')
    cax.xaxis.set_label_position('top')
pd.DataFrame(stats).to_csv(ROOT / 'results/shape_rrl_fit_stats.csv', index=False, float_format='%.4f')
fig.savefig(ROOT / 'figures/paper_shape_rrl_residuals.png')

In [ ]:
from IPython.display import Image
Image(filename='figures/paper_shape_rrl_residuals.png')

**Figure 8.4.** Residuals of the oblate model in the RR Lyrae sky cells, for the best fit (a, b) and at the
inclination of the metal-poor giants (c, d): (a, c) Pearson residuals of the counts, $(n-\lambda)/\sqrt\lambda$; (b, d) median
distance modulus minus model (mag). Solid line: the fit's line of nodes; dashed: the kinematic one. East to the left.
`scripts/paper_shape_rrl_residuals.py`